In [1]:
#install if missing
!pip install torchmetrics

In [6]:
!pip install watermark

  Obtaining dependency information for watermark from https://files.pythonhosted.org/packages/67/ee/32fec350c20ff060064d058c37df53dd1fad74dfea9532faf2868b0512b5/watermark-2.4.3-py2.py3-none-any.whl.metadata


In [10]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage import io, transform, color, exposure, img_as_float32
import torch
from torch import nn
from torchmetrics.classification import MultilabelAccuracy
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader, random_split
from CNNDataset import CNNDataset
%matplotlib inline

In [8]:
%load_ext watermark

In [11]:
%watermark --iversions 

matplotlib: 3.7.2
numpy     : 1.24.3
pandas    : 2.0.3
skimage   : 0.20.0
torch     : 2.1.0+cu118



In [3]:
path_to_images = "./ocular-disease-recognition-odir5k/"
training_path = "ODIR-5k/ODIR-5k/Training Images/"

dataset = CNNDataset("Processed_data.xlsx", path_to_images + training_path)

In [4]:
# Specify the ratio of training to testing data
train_ratio = 0.8
dataset_size = len(dataset)
train_size = int(train_ratio * dataset_size)
test_size = dataset_size - train_size

In [5]:
# Use random_split to create training and testing datasets
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoader for training and testing
train_loader = DataLoader(train_dataset, batch_size=14, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=14, shuffle=False, num_workers=2)

In [6]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [7]:
class EyesCNN(nn.Module):
    
    def __init__(self, dropout_rate=0.5):
        super(EyesCNN, self).__init__()
     
        self.conv1 = nn.Conv2d(3, 32, 3, 1, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)
        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1)
        self.conv4 = nn.Conv2d(128, 128, 3, 1, 1)
        self.conv5 = nn.Conv2d(128, 256, 2, 2, 1)
        self.conv6 = nn.Conv2d(256, 128, 2, 2, 1)
        
        # Define the max pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        # Define batch normalization layer
        self.bn = nn.BatchNorm2d(128)

        # Define dropout layer
        self.dropout = nn.Dropout2d(p=dropout_rate)

        # Define fully connected layers
        self.fc1 = nn.Linear(128 * 16 * 16 + 1, 128)  # +1 for the age input
        self.fc2 = nn.Linear(128, 8)

    def forward(self, x, age):
        # Apply depthwise separable convolution and pooling layers
        x = F.leaky_relu(self.conv1(x))
        x = F.leaky_relu(self.conv2(x))
        x = F.leaky_relu(self.conv3(x))
        x = F.leaky_relu(self.conv4(x))
        x = F.leaky_relu(self.conv4(x))
        x = F.leaky_relu(self.conv5(x))
        x = F.leaky_relu(self.conv6(x))
        x = self.bn(x)
        x = self.pool(x)

        # Apply dropout
        x = self.dropout(x)

        # Reshape the tensor for fully connected layers
        x = x.view(x.size(0), -1)  # The size of the first dimension might vary (batch size)

        # Concatenate age information to the flattened tensor
        x = torch.cat((x, age.view(-1, 1)), dim=1)

        # Apply fully connected layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = F.sigmoid(x)

        return x

In [8]:
# Create an instance of the SeparableCNN
model = EyesCNN().to(device)

# Print the model architecture
print(model)

EyesCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv5): Conv2d(128, 256, kernel_size=(2, 2), stride=(2, 2), padding=(1, 1))
  (conv6): Conv2d(256, 128, kernel_size=(2, 2), stride=(2, 2), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout2d(p=0.5, inplace=False)
  (fc1): Linear(in_features=32769, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=8, bias=True)
)


In [9]:
# Set loss function and optimizer
criterion = torch.nn.BCELoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.003)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)  # Adjust step_size and gamma as needed

In [ ]:
# Training loop
num_epochs = 15  # Set the number of training epochs
accuracy_metric = MultilabelAccuracy(num_labels = 8).to(device)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    print("Training Now: ")
    
    for inputs, age, labels in train_loader:
        # Move inputs, and labels to device
        inputs, age, labels = inputs.float().to(device), age.float().to(device), labels.float().to(device)
        
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs, age).float()
        
        # Compute the loss
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        #Increment Loss
        running_loss += loss.item()
        
        # clear cuda cache for efficient memory usage.
        del inputs, age, labels
        torch.cuda.empty_cache()
        
        
    # Calculate multi-label accuracy
    model.eval()  # Set the model to evaluation mode
    all_outputs, all_labels = [], []
    
    with torch.no_grad():
        for inputs, age, labels in test_loader:
            inputs, age, labels = inputs.float().to(device), age.float().to(device), labels.float().to(device)
            outputs = model(inputs, age).float()
            all_outputs.append(outputs)
            all_labels.append(labels)
            
            del inputs, age, labels
            torch.cuda.empty_cache()
            

    all_outputs = torch.cat(all_outputs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    accuracy = accuracy_metric(all_outputs, all_labels)
    
    # Print average loss and accuracy for the epoch
    average_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {average_loss:.4f}, Multi-label Accuracy: {accuracy.item() * 100:.2f}%")
    
print("Training finished.")

Training Now: 
Epoch [1/15], Loss: 0.4032, Multi-label Accuracy: 85.53%
Training Now: 
Epoch [2/15], Loss: 0.3286, Multi-label Accuracy: 85.95%
Training Now: 
Epoch [3/15], Loss: 0.3345, Multi-label Accuracy: 86.26%
Training Now: 
Epoch [4/15], Loss: 0.3258, Multi-label Accuracy: 86.26%
Training Now: 
Epoch [5/15], Loss: 0.3266, Multi-label Accuracy: 86.26%
Training Now: 
Epoch [6/15], Loss: 0.3440, Multi-label Accuracy: 86.26%
Training Now: 
Epoch [7/15], Loss: 0.3288, Multi-label Accuracy: 86.26%
Training Now: 
Epoch [8/15], Loss: 0.3257, Multi-label Accuracy: 86.26%